# Bootstrap CI cho median thời gian giao hàng theo bang

Thời gian giao (ngày, từ lúc mua tới lúc khách nhận) lệch phải: vài đơn mất hơn 100 ngày. Báo cáo
**median** theo bang bền hơn mean, nhưng median không có công thức SE gọn như mean. Bài này dựng CI
cho median bằng hai cách, **bootstrap percentile** và **CI chính xác từ thống kê thứ tự** (order
statistic, không cần giả định phân phối), rồi so chúng trên các bang nhiều đơn và ít đơn.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) là chỗ đọc dữ liệu thật, vẽ, gọi hàm. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/02-probability-statistics/exercises/intermediate/02-bootstrap-median -q`

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
dates = ["order_purchase_timestamp", "order_delivered_customer_date"]
orders = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")

delivered = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
df = delivered.merge(customers[["customer_id", "customer_state"]], on="customer_id")
df["days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp).dt.total_seconds() / 86400
df.days.describe(percentiles=[0.5, 0.95, 0.99])

In [ ]:
# explore: số đơn mỗi bang chênh nhau gần 1000 lần: SP hơn 40 nghìn, RR vài chục
df.customer_state.value_counts().agg(["max", "min"])

## 1. Phân phối bootstrap

`bootstrap_stats(x, stat=np.median, n_boot=2000, seed=0)` trả mảng `n_boot` giá trị thống kê, mỗi
giá trị tính trên một mẫu lấy **có hoàn lại**, cùng cỡ với `x`. `stat` là hàm numpy nhận tham số
`axis` (`np.median`, `np.mean`…): vector hoá bằng ma trận chỉ số rồi gọi `stat(..., axis=1)`.

Với `x` lớn (96 nghìn đơn × 2000 lần là 192 triệu số, khoảng 1.5 GB), chia `n_boot` thành nhiều
**lô** sao cho mỗi lô không quá khoảng 5 triệu phần tử. Dùng một `rng = np.random.default_rng(seed)`
cho mọi lô để kết quả tái lập được.

In [ ]:
def bootstrap_stats(x: np.ndarray, stat=np.median, n_boot: int = 2000, seed: int = 0) -> np.ndarray:
    """n_boot giá trị stat trên các mẫu bootstrap của x."""
    # TODO
    raise NotImplementedError

## 2. Hai cách dựng CI cho median

- `percentile_ci(x, stat=np.median, n_boot=2000, seed=0, alpha=0.05)`: percentile `alpha/2` và
  `1 − alpha/2` của phân phối bootstrap (dùng `np.quantile`, nội suy mặc định).
- `median_ci_exact(x, alpha=0.05)`: gọi B là số quan sát nằm dưới median thật, B ~ Binomial(n, 0.5).
  Với `j = int(stats.binom.ppf(alpha / 2, n, 0.5))` (tối thiểu 1), khoảng
  `[x_(j), x_(n−j+1)]` (thống kê thứ tự, đánh số từ 1) chứa median thật với xác suất ít nhất
  `1 − alpha`, **với mọi phân phối liên tục**. Trong numpy: `xs[j - 1]` và `xs[n - j]` với
  `xs = np.sort(x)`.

In [ ]:
def percentile_ci(x: np.ndarray, stat=np.median, n_boot: int = 2000, seed: int = 0,
                  alpha: float = 0.05) -> tuple[float, float]:
    """CI bootstrap percentile cho stat."""
    # TODO
    raise NotImplementedError


def median_ci_exact(x: np.ndarray, alpha: float = 0.05) -> tuple[float, float]:
    """CI cho median từ thống kê thứ tự (phân phối Binomial(n, 0.5))."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: toàn bộ đơn giao xong. Hai cách cho gần như cùng một khoảng khi n lớn
x = df.days.to_numpy()
print("median", np.median(x))
print("bootstrap", percentile_ci(x))
print("exact    ", median_ci_exact(x))

## 3. Bảng theo nhóm

`ci_table(df, value, group, min_n=1, n_boot=2000, seed=0)`: một dòng cho mỗi nhóm có ít nhất `min_n`
quan sát; index là tên nhóm; cột theo đúng thứ tự `n, median, boot_lo, boot_hi, exact_lo, exact_hi`;
sắp theo `median` giảm dần. Mỗi nhóm gọi `percentile_ci(values, np.median, n_boot, seed)` với
**cùng** `seed`.

In [ ]:
def ci_table(df: pd.DataFrame, value: str, group: str, min_n: int = 1, n_boot: int = 2000,
             seed: int = 0) -> pd.DataFrame:
    """Median và hai loại CI cho từng nhóm."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: bảng theo bang. Bang ít đơn có CI rộng tới hơn 10 ngày
table = ci_table(df, "days", "customer_state")
table["width"] = table.boot_hi - table.boot_lo
table.round(2)

In [ ]:
# explore: vẽ median và CI, bang sắp theo median
import matplotlib.pyplot as plt

t = table.sort_values("median")
plt.figure(figsize=(6, 7))
plt.errorbar(t["median"], t.index, xerr=[t["median"] - t.boot_lo, t.boot_hi - t["median"]], fmt="o")
plt.xlabel("median số ngày giao (CI 95% bootstrap)")
plt.show()

## 4. Bootstrap hỏng khi nào

Bootstrap giả định "mẫu là bản thu nhỏ của tổng thể". Với thống kê phụ thuộc vào vài quan sát cực
trị (max, percentile 99 trên mẫu nhỏ), giả định đó vỡ. Chạy cell dưới: coi toàn bộ đơn là tổng thể,
lấy mẫu 200 đơn, dựng CI bootstrap cho **max** và đếm xem bao nhiêu lần nó chứa max thật.

In [ ]:
# explore: CI bootstrap cho max gần như không bao giờ chứa max thật
rng = np.random.default_rng(0)
true_max = x.max()
hits = 0
for r in range(200):
    sample = rng.choice(x, 200)
    lo, hi = percentile_ci(sample, np.max, n_boot=500, seed=r)
    hits += lo <= true_max <= hi
print("độ phủ của CI bootstrap cho max:", hits / 200)

## 5. Nhận xét

Viết 4–6 câu: bang nào chậm nhất và bạn chắc cỡ nào; vì sao không nên xếp hạng bang chỉ theo median
điểm; hai loại CI khác nhau ở đâu, khi nào chọn cái nào; và vì sao bootstrap cho max thất bại.